# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [ ]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

In [ ]:
import json

In [ ]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [ ]:
type(train_data)

In [ ]:
list(train_data.keys())

In [ ]:
type(train_data["data"])

In [ ]:
len(train_data["data"])

In [ ]:
type(train_data["data"][0])

In [ ]:
train_data["data"][0].keys()

In [ ]:
train_data["data"][0]["title"]

In [ ]:
len(train_data["data"][0]["paragraphs"])

In [ ]:
train_data["data"][0]["paragraphs"][0]

In [ ]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [ ]:
# YOUR CHANGES HERE
import pandas as pd
parsed_rows = [] #store in dict
for d in train_data['data']:
    title = d['title']
    for p_idx, paragraph in enumerate(d['paragraphs']):
        parsed_rows.append({
            'document_title': title,
            'paragraph_index': p_idx,
            'paragraph_context': paragraph['context']
        })
#save in df
parsed_df = pd.DataFrame(parsed_rows)
parsed_df

Submit "parsed.tsv" in Gradescope.

In [ ]:
parsed_df.to_csv('parsed.tsv', sep='\t', index=False, encoding='utf-8', quotechar='"')

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [ ]:
# YOUR CHANGES HERE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [ ]:
# YOUR CHANGES HERE
import numpy as np
import gc

import gc
import os

#check memory to see that there is enough space for svd
def mem():
    print(os.popen('free -h').read())

parsed_df = pd.read_csv('parsed.tsv', sep='\t')
mem()

In [ ]:
#fit TF-IDF on all paragraph texts
vectorizer = TfidfVectorizer(max_features=50000, sublinear_tf=True)
tfidf_matrix = vectorizer.fit_transform(parsed_df['paragraph_context'])
mem()

In [ ]:
#reduce dimension via SVD
svd = TruncatedSVD(n_components=1024, random_state=42, algorithm='randomized', n_iter=2)
paragraph_vectors = svd.fit_transform(tfidf_matrix)
del tfidf_matrix
gc.collect() #release unused memory for storage
mem()

In [ ]:
#L2-normalize vectors
paragraph_vectors = normalize(paragraph_vectors, norm='l2')
mem()

In [ ]:
#save output in df
parsed_df['paragraph_vector_json'] = [
    json.dumps(vec.tolist()) for vec in paragraph_vectors
]
mem()

In [ ]:
#save output in df
output_df = parsed_df[['document_title', 'paragraph_index', 'paragraph_vector_json']]
print(output_df)
mem()

In [ ]:
#save as tsv.gz
output_df.to_csv('paragraph-vectors.tsv.gz', sep='\t', index=False, compression='gzip')

Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [ ]:
# YOUR CHANGES HERE
#load questions file
questions_df = pd.read_csv('questions.tsv', sep='\t')
#transform using same vectorizer and svd from Part 3
question_tfidf = vectorizer.transform(questions_df['question'])
question_vectors = svd.transform(question_tfidf)
question_vectors = normalize(question_vectors, norm='l2')

#save output in df
questions_df['question_vector_json'] = [json.dumps(vec.tolist()) for vec in question_vectors]
output_df = questions_df[['question_id', 'question_vector_json']]
output_df

Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [ ]:
# YOUR CHANGES HERE
output_df.to_csv('question-vectors.tsv', sep='\t', index=False)

Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [ ]:
# YOUR CHANGES HERE
from sklearn.neighbors import NearestNeighbors

#fit nearest neighbors on paragraph vectors
nn = NearestNeighbors(n_neighbors=5, metric='euclidean')
nn.fit(paragraph_vectors)

#find top 5 nearest paragraphs for each question
distances, indices = nn.kneighbors(question_vectors)

#build output rows
rows = []
for q_pos, question_id in enumerate(questions_df['question_id']):
    for rank, para_idx in enumerate(indices[q_pos]):
        rows.append({
            'question_id': question_id,
            'question_rank': rank + 1,  # 1-indexed rank
            'document_title': parsed_df.iloc[para_idx]['document_title'],
            'paragraph_index': parsed_df.iloc[para_idx]['paragraph_index']
        })

#save to df
matches_df = pd.DataFrame(rows)
matches_df

Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [ ]:
# YOUR CHANGES HERE
matches_df.to_csv('question-matches.tsv', sep='\t', index=False)

Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [ ]:
#get the first 5 questions sorted by question_id ascending
first_5_ids = sorted(questions_df['question_id'].unique())[:5]

worst_rows = []
for question_id in first_5_ids:
    #get the 5 matches for this section, ranked 1-5
    q_matches = matches_df[matches_df['question_id'] == question_id].sort_values('question_rank')
    
    #print question and all matched paragraphs for review
    question_text = questions_df[questions_df['question_id'] == question_id]['question'].values[0]
    print(f"\nQuestion ID: {question_id}")
    print(f"Question: {question_text}")
    print("Matched paragraphs:")
    
    for _, row in q_matches.iterrows():
        para_text = parsed_df[
            (parsed_df['document_title'] == row['document_title']) & 
            (parsed_df['paragraph_index'] == row['paragraph_index'])
        ]['paragraph_context'].values[0]
        print(f"\n  Rank {row['question_rank']} | {row['document_title']} | para {row['paragraph_index']}:")
        print(f"  {para_text[:200]}...")  # print first 200 chars for readability

In [ ]:
#save worst paragraphs
worst_rows = [
    {'question_id': 1,  'document_title': 'Tuvalu',        'paragraph_index': 49},
    {'question_id': 4,  'document_title': 'BeiDou_Navigation_Satellite_System', 'paragraph_index': 3},
    {'question_id': 7,  'document_title': 'Beyoncé',       'paragraph_index': 24},
    {'question_id': 10, 'document_title': 'Roman_Republic', 'paragraph_index': 48},
    {'question_id': 13, 'document_title': 'San_Diego',     'paragraph_index': 16},
]
#save in df
worst_df = pd.DataFrame(worst_rows)
worst_df

In [ ]:
#save in tsv
worst_df.to_csv('worst-paragraphs.tsv', sep='\t', index=False)

## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [ ]:
with open('acknowledgments.txt', 'w') as f:
    f.write("When working on this project, I references the following websites:")
    f.write("https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html")
    f.write("https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html")
    f.write("https://medium.com/swlh/truncated-singular-value-decomposition-svd-using-amazon-food-reviews-891d97af5d8d")